# 💊 Cheminformatics: Molecular Fundamentals

This notebook introduces the basics of working with molecules in Python using **RDKit**.

**You will learn:**
1. Parse molecules from SMILES strings
2. Visualize 2D structures
3. Compute molecular descriptors
4. Apply Lipinski's Rule of Five (drug-likeness)
5. Generate molecular fingerprints


## 1. Setup

Install RDKit: `pip install rdkit` or `conda install -c conda-forge rdkit`.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, Descriptors, Lipinski
from rdkit.Chem.Draw import IPythonConsole
import pandas as pd

print("RDKit ready.")


## 2. SMILES — molecules as strings

**SMILES** (Simplified Molecular Input Line Entry System) encodes a molecule as a string. Examples:

| Drug | SMILES |
|------|--------|
| Aspirin | `CC(=O)Oc1ccccc1C(=O)O` |
| Caffeine | `CN1C=NC2=C1C(=O)N(C(=O)N2C)C` |
| Ibuprofen | `CC(C)Cc1ccc(cc1)C(C)C(=O)O` |
| Paracetamol | `CC(=O)Nc1ccc(O)cc1` |


In [ ]:
# Common drugs in SMILES
drugs = {
    "Aspirin":     "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine":    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Ibuprofen":   "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Paracetamol": "CC(=O)Nc1ccc(O)cc1",
    "Atorvastatin":"CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CC[C@@H](O)C[C@@H](O)CC(=O)O",
}

# Convert each SMILES to a Mol object
molecules = {name: Chem.MolFromSmiles(smi) for name, smi in drugs.items()}

# Verify they all parsed correctly
for name, mol in molecules.items():
    print(f"{name:15} → {mol.GetNumAtoms()} heavy atoms, {mol.GetNumBonds()} bonds")


## 3. Visualization

RDKit can draw molecules nicely in Jupyter.

In [ ]:
# Draw a single molecule
aspirin = molecules["Aspirin"]
Draw.MolToImage(aspirin, size=(300, 300))


In [ ]:
# Draw all of them in a grid
img = Draw.MolsToGridImage(
    list(molecules.values()),
    legends=list(molecules.keys()),
    molsPerRow=3,
    subImgSize=(250, 250)
)
img


## 4. Molecular descriptors

A **descriptor** is a numerical property computed from a structure. They are the basis of QSAR and ADMET prediction.

The most common ones:

- **Molecular Weight (MW)**
- **LogP**: octanol-water partition coefficient (lipophilicity)
- **HBD / HBA**: hydrogen bond donors / acceptors
- **TPSA**: topological polar surface area
- **Rotatable bonds**: molecular flexibility


In [ ]:
def compute_descriptors(name, mol):
    return {
        "Drug": name,
        "MW":              round(Descriptors.MolWt(mol), 2),
        "LogP":            round(Descriptors.MolLogP(mol), 2),
        "HBD":             Lipinski.NumHDonors(mol),
        "HBA":             Lipinski.NumHAcceptors(mol),
        "TPSA":            round(Descriptors.TPSA(mol), 2),
        "RotatableBonds":  Lipinski.NumRotatableBonds(mol),
        "AromaticRings":   Lipinski.NumAromaticRings(mol),
    }

df = pd.DataFrame([compute_descriptors(n, m) for n, m in molecules.items()])
df


## 5. Lipinski's Rule of Five

Christopher Lipinski (Pfizer, 1997) noticed that orally active drugs tend to satisfy:

- MW ≤ 500 Da
- LogP ≤ 5
- HBD ≤ 5
- HBA ≤ 10

A molecule that violates ≥ 2 rules is unlikely to be a good oral drug. It is a **rapid filter for druglikeness**.

> ⚠️ It is a guideline, not an absolute rule. Many real drugs violate it (cyclosporine, antibiotics, etc.).


In [ ]:
def lipinski_violations(mol):
    violations = 0
    if Descriptors.MolWt(mol)      > 500: violations += 1
    if Descriptors.MolLogP(mol)    > 5:   violations += 1
    if Lipinski.NumHDonors(mol)    > 5:   violations += 1
    if Lipinski.NumHAcceptors(mol) > 10:  violations += 1
    return violations

df["LipinskiViolations"] = [lipinski_violations(m) for m in molecules.values()]
df["DrugLike"] = df["LipinskiViolations"] <= 1
df


## 6. Molecular fingerprints

A **fingerprint** is a binary vector encoding the presence of substructures in a molecule. Used to:

- Compare similarity between molecules (Tanimoto coefficient)
- Feed ML models (each bit = a feature)
- Substructure search

The most popular family is **Morgan / ECFP** (circular).


In [ ]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

# Modern Morgan generator (radius=2 ≈ ECFP4)
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# Compute fingerprints for all molecules
fps = {name: mfpgen.GetFingerprint(mol) for name, mol in molecules.items()}

# Pairwise similarity (Tanimoto)
print("Tanimoto similarity between drugs:\n")
print(f"{'':<15}", *[f"{n:<13}" for n in molecules.keys()])
for n1, fp1 in fps.items():
    sims = [DataStructs.TanimotoSimilarity(fp1, fp2) for fp2 in fps.values()]
    print(f"{n1:<15}", *[f"{s:<13.3f}" for s in sims])


## 📝 Exercises

1. **Easy**: add 5 more drugs you know and recompute the descriptor table.
2. **Medium**: write a function that takes a SMILES and returns whether it passes Lipinski's Rule of Five.
3. **Medium**: search for the 3 most similar drugs to aspirin in your set (Tanimoto).
4. **Hard**: download a list of 100 FDA-approved drugs from DrugBank or ChEMBL and analyze descriptor distributions.
5. **Pharma-focused**: pick a drug class you know (e.g. NSAIDs) and show that they cluster together by molecular similarity.

## 📚 Resources

- [RDKit Documentation](https://www.rdkit.org/docs/)
- [Lipinski's Rule of Five (paper)](https://doi.org/10.1016/S0169-409X(96)00423-1)
- [SMILES Tutorial - Daylight](https://www.daylight.com/dayhtml/doc/theory/theory.smiles.html)
